# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

# %matplotlib widget

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import hist
import mplhep as hep


from rich import print
from matplotlib.colors import LogNorm
from dunetrg.analysis.histograms import cumsum_hist, cut_scan_to_df, compute_regaxis_specs

from dunetrg.analysis.viz.tps import TrgPrimitivesPlotter
from dunetrg.analysis.viz.backtracker import BackTrackerPlotter
from dunetrg.analysis.viz.display import TriggerPrimitivesEventViewer
from dunetrg.analysis.viz.textual import dataframe_to_rich_table
from typing import Literal


# Data

In [ ]:
import dunetrg.data.datacatalogue as dctl

dataset_names = ['eminus', 'radbkg']
datasets = dctl.load('vd/1x8x14/3sig', dataset_names)
rad_ws=datasets['radbkg']
em_ws=datasets['eminus']


# Analysers

In [ ]:
class RadBkgCrpRateAnalyser:
    """Analyses radioactive background TP rates per CRP as a function of adc_peak threshold.

    Computes trigger primitive rates separately for backtracked background deposits
    and untracked noise TPs, scanning over a sequence of adc_peak cuts to characterise
    the threshold dependence of each readout plane's rate.
    """

    rop_alias = {
        0: 'U',
        1: 'V',
        2: 'X'
    }

    fmt_adc_threshold = {
        'adc_peak_min': '>= {:.0f}'
    }

    def __init__(self, rad_ws):
        """
        Parameters
        ----------
        rad_ws:
            Radioactive background TriggerAnalysisWorkspace.
        """
        self._rad_ws = rad_ws
        self.adc_peak_cuts = list(range(28,60,2))



    def do_total_rates(self):
        """Print a colour-coded table and plot of total CRP TP rates vs adc_peak threshold.

        Rates (Hz/CRP) are colour-coded: green < 100 kHz, yellow 100–150 kHz, red > 150 kHz.
        """

        rad_ws = self._rad_ws
        tpp = TrgPrimitivesPlotter(rad_ws)

        h_cs = tpp.make_cutsequence_hist('adc_peak', self.adc_peak_cuts)
        h_cs_tot = h_cs[:, sum, :]

        def col_crp_rate(r):
            if r > 1.5e5:
                col = 'red'
            elif r < 1e5:
                col = 'green'
            else:
                col = 'yellow'

            r_str = f'{r:.2f} Hz'
            if col:
                r_str = f'[{col}]{r_str}[/{col}]'
            return r_str
                



        h_rate_crp = h_cs_tot/tpp.simulated_time()/tpp.geo.num_crps

        rate_tot = cut_scan_to_df(h_rate_crp, 'readout_plane_id', 'adc_peak')

        # Cosmetic
        _fmts={
            f'readout_plane_id_{rop}': col_crp_rate for rop in range(3)
        }
        _fmts.update(self.fmt_adc_threshold)

        # TODO: generalize        
        _col_labels = {
            'adc_peak_min': 'threshold'
        }
        _col_labels.update({
            f'readout_plane_id_{rop}': f'{self.rop_alias[rop]} [{rop}]' for rop in range(3)
        })
        
        t_rate_tot = dataframe_to_rich_table(
            rate_tot,
            formatters=_fmts,
            column_titles=_col_labels)
        t_rate_tot.title = "CRP rates vs TP threshold"
        print(t_rate_tot)

        # TODO: move to separate function
        fig, axes = plt.subplots(1,3, figsize=(15,4))
        for rop_id in range(3):
            _ax=axes[rop_id]
            hep.histplot(h_rate_crp[rop_id,:], ax=_ax)

            _ax.set_yscale('log')
            _ax.set_title(f"RO Plane {rop_id}")
            _ax.axhline(1.5e5, c='darkorange')
            _ax.axhline(1e5, c='red')

        fig.tight_layout()

    

    def make_cut_scan(self):
        """Build a DataFrame of per-ROP background and noise CRP rates vs adc_peak threshold.

        TPs backtracked to a physics deposit are labelled 'bkg'; untracked TPs are 'noi'.

        Returns
        -------
        pd.DataFrame
            Columns: adc_peak_min, rate_rop_{0,1,2}_bkg, rate_rop_{0,1,2}_noi (all in Hz/CRP).
        """

        rad_ws = self._rad_ws
        tpp = TrgPrimitivesPlotter(rad_ws)

        h_cs = tpp.make_cutsequence_hist('adc_peak', self.adc_peak_cuts)
        h_cs_bkg = h_cs[:, 1j, :]
        h_cs_noi = h_cs[:, 0j, :]
        # h_cs_tot = h_cs[:, sum, :]
        # h_rate_crp = h_cs_tot/tpp.simulated_time()/tpp.geo.num_crps


        h_rate_bkg_crp = h_cs_bkg/tpp.simulated_time()/tpp.geo.num_crps
        h_rate_noi_crp = h_cs_noi/tpp.simulated_time()/tpp.geo.num_crps


        cut_scan_bkg = cut_scan_to_df(h_rate_bkg_crp, 'readout_plane_id', 'adc_peak')
        cut_scan_noi = cut_scan_to_df(h_rate_noi_crp, 'readout_plane_id', 'adc_peak')


        cut_scan_bkg.rename(columns={
            f'readout_plane_id_{i}':f'rate_rop_{i}' for i in range(3)
        }, inplace=True)
        cut_scan_noi.rename(columns={
            f'readout_plane_id_{i}':f'rate_rop_{i}' for i in range(3)
        }, inplace=True)

        cut_scan = pd.merge(cut_scan_bkg, cut_scan_noi, on="adc_peak_min", suffixes=('_bkg', '_noi'))

        return cut_scan

    def do_purity_tables(self):
        """Print per-ROP tables of background vs noise CRP rates and their ratio vs adc_peak threshold."""

        _fmts={
            f"rate_rop_{rop}_{ds}": '{:0.2f} Hz' for rop in range(3) for ds in ('bkg', 'noi')
        }

        _fmts.update({
            'ratio': '{:.2f}'
        })

        cut_scan=self.make_cut_scan()

        for i in range(3):
            cut_scan_rop = cut_scan[["adc_peak_min", f'rate_rop_{i}_bkg', f'rate_rop_{i}_noi']]
            cut_scan_rop['ratio'] = cut_scan[f'rate_rop_{i}_noi']/cut_scan[f'rate_rop_{i}_bkg']
            print(dataframe_to_rich_table(cut_scan_rop, formatters=_fmts, show_index=True, title=f'Background vs Noise CRP rates (ROP {i})'))

    def do_corrected_purity_tables(self):
        """Print purity tables with noise rates taken from the collection plane for all planes.

        Induction-plane noise counts are substituted with the collection-plane (ROP 2) noise,
        correcting for the known mistagging of background TPs as noise on induction planes.
        """

        _fmts={
            f"rate_rop_{rop}_{ds}": '{:0.2f} Hz' for rop in range(3) for ds in ('bkg', 'noi')
        }

        _fmts.update({
            'ratio': '{:.2f}'
        })
        
        cut_scan=self.make_cut_scan()

        cut_scan[f'rate_rop_0_bkg'] = cut_scan[f'rate_rop_0_bkg']+cut_scan[f'rate_rop_0_noi']-cut_scan[f'rate_rop_2_noi']
        cut_scan[f'rate_rop_1_bkg'] = cut_scan[f'rate_rop_1_bkg']+cut_scan[f'rate_rop_1_noi']-cut_scan[f'rate_rop_2_noi']
        cut_scan[f'rate_rop_0_noi'] = cut_scan[f'rate_rop_2_noi']
        cut_scan[f'rate_rop_1_noi'] = cut_scan[f'rate_rop_2_noi']


        for i in range(3):
            cut_scan_rop = cut_scan[["adc_peak_min", f'rate_rop_{i}_bkg', f'rate_rop_{i}_noi']]
            cut_scan_rop['ratio'] = cut_scan[f'rate_rop_{i}_noi']/cut_scan[f'rate_rop_{i}_bkg']
            print(dataframe_to_rich_table(cut_scan_rop, formatters=_fmts, show_index=True, title=f'Background vs Noise CRP rates (ROP {i})'))

# Trigger Primitive Rates vs `adc_peak`
---

Calculate the trigger primitives rate as a function of the `adc_peak`  TP threshold.

Highlight regions below 100 kHz (green), between 100 kHz and 150 kHz (yellow), and above 150 kHz (red)

In [ ]:
ana = RadBkgCrpRateAnalyser(rad_ws=rad_ws)
ana.do_total_rates()

### Observations
- The TP rates thrends differ significantly between the collection plane and the induction planes.
- The induction plane rates drop faster than the collection rate.
- The 'target' rate of 100 kHz is reached only at `adc_peak==50` on the collection plane, whereas it is reach at `adc_peak==44 and 40` for the two induction planes
  
### Conclusions
- For MC production, the threshold selection target is ~150 kHz background rate per plane.
- This translates to 
  - I0: 38
  - I2: 38
  - C: 40 (or 38 for simplicity)


# Impact of increasing TP threshold on e- backtracking efficiency 
---

Question: what fraction of e- signal is lost at the increase of the TP threshold?
- Sample: e-minus, 100 events
- Energy: 1-100 MeV
The e- sample is choosen since e- are the reference interaction in low-E regions

Steps:
- Plot KE distribution to confirm sufficient presence of low energy events
- Calculate the relative loss of backtracking efficiency as a function of adc_peak cut
  - Use relative loss since plane efficiencis are different
  - Plot the evolution of relative efficiency loss vs threshold
  - Confirm that losses in the are negligible for e- event

---

In [ ]:
h = hist.Hist(hist.axis.Regular(10, 0, 100, underflow=True, overflow=True, name="ke", label='Kinetic Energy [MeV]'))

h.fill((em_ws.mctruths.kinetic_energy*1000))

fig, ax = plt.subplots(1,1)
hep.histplot(h, histtype='step', ax=ax)

ax.grid()
ax.set_ylabel('Counts')

# (em_ws.mctruths.kinetic_energy*1000).hist(bins=25)


In [ ]:
btp = BackTrackerPlotter(em_ws)
evw = TriggerPrimitivesEventViewer(em_ws)

### Completeness (Backtracking efficiency) across planes for single $e^{-}$ events [1-100 MeV]

In [ ]:
fig = btp.draw_nel_eff_by_plane(bins=20)

**Notable features**: 
- A small fraction of events are effectively not captured by TPs (eff < 0.1>)
- They are present on all planes
- As usual, induction planes are less effective in capturing the drift electron information

### Impact of incremental `adc_peak` cuts on completeness

In [ ]:
ke_cut = "kinetic_energy < 20/1000"
# ke_cut = "kinetic_energy > 50/1000"

adc_cutsequence = np.arange(26,61,2)

eff_base = btp.compute_nel_eff_by_plane()
ke = em_ws.mctruths[['event_uid', 'kinetic_energy']]
eff_base = eff_base.merge(ke, on='event_uid')
eff_base.query(ke_cut, inplace=True)

# quality cut: if collection efficiency is 0, the event was not visible
# eff_base.query("nel_eff_rop2 > 0", inplace=True)

eff_base_m = eff_base.mean()
display(pd.DataFrame(eff_base_m[['nel_eff_rop0', 'nel_eff_rop1', 'nel_eff_rop2']]))


rows = []


for adc_peak_min in adc_cutsequence:
    eff_cut = btp.compute_nel_eff_by_plane(f"adc_peak > {adc_peak_min}")
    eff_cut = eff_cut.merge(ke, on='event_uid')
    eff_cut.query(ke_cut, inplace=True)

    # quality cut: if collection efficiency is 0, the event was not visible
    # eff_cut.query("nel_eff_rop2 > 0", inplace=True)

    m = (eff_cut.mean()/eff_base_m)
    m= m.drop(['event_uid', 'kinetic_energy'])
    
    m['adc_peak_min'] = adc_peak_min

    rows.append(m)

x = pd.DataFrame(rows)
x.set_index('adc_peak_min', inplace=True)

_fmts={
    f"nel_eff_rop{rop}":'{:.2f}' for rop in range(3)
}

t = dataframe_to_rich_table(x, formatters=_fmts, show_index=True, index_name='adc_peak_min')
t.title = 'Single e- relative efficiency (n_el) vs threhshold'
print(t)

fig, ax = plt.subplots(1,1)

x.plot(ax=ax)
ax.set_title('Single $e^{-}$ (1-20 MeV) relative efficiency ($N^{el}$) vs threhshold')




### Observations

The increase in `adc_peak` threshold has a negligible impact on the ability to single $e^{-}$ completeness.

# What is the Noise/Backgrounds TP composition vs threshold?

- What is the fraction of backgrounds TPs vs noise?
- Is it compatible with expectations?
- If not:
  - Is the estimate reliable?
  - What is the cause?

In [ ]:
bkg_ana = RadBkgCrpRateAnalyser(rad_ws)

In [ ]:
bkg_ana.do_purity_tables()

### Observations

- The bkg/noise ratio (purity) rapidly falls to zero as expected only for the collection plane
- The purity settles to 10-12% for induction planes
- Based on previous experience, this is likey to be related to TPs being erroneously flagged as noise on the induction planes

### Trigger Primitive distribution inspection

The separation of TPs in noise and signal depends on backtracking and the presence of `SimChannels` in detsim products.
Is is know that in radioactive background samples, high-energy noise TPs are found, compatible with background deposits  even if none is found


## TP backgrounds/noise distributions

In [ ]:

rad_tpp = TrgPrimitivesPlotter(rad_ws)

vars = {
    'adc_peak': {'bin_size': 10, 'xlim': (0, 400)},
    'samples_over_threshold': {'bin_size': 1, 'xlim': (None, 20)},
    'adc_integral': {'bin_size': 10, 'xlim': (0, 2000)},
}
for v, cfg in vars.items():
    h = rad_tpp.make_var_hist(v, cfg['bin_size'])

    fig, axes = plt.subplots(1,3, figsize=(12,4))

    for rop_id in range(3):
        ax=axes[rop_id]

        hep.histplot(h[rop_id,sum,:], ax=ax, yerr=False, color='black', label='Total')
        hep.histplot(h[rop_id,1,:], ax=ax, yerr=False, histtype='fill', label='RadBkg', alpha=0.75)
        hep.histplot(h[rop_id,0,:], ax=ax, yerr=False, histtype='fill', label='Noise', alpha=0.75)
        ax.set_yscale('log')
        ax.set_xlim(*cfg['xlim'])
        ax.grid()
        ax.legend()
        ax.set_ylabel('Counts')
        
        ax.set_title(f"Readout plane {rop_id}")
    fig.suptitle(f"{v} distributions")

    fig.tight_layout()

### Observations


- All distributions of TP variables flagged as 'noise' (not backtracked to a deposit) show a tail at high values
- The tail is more pronounced in induction planes
- The noise profile used in WireCell, the same across planes [_to my understanding_]
  - This supported by raw adc distributions from electron gun samples
- Inspection of noise TPs with high `adc_peak`/`samples_over_threshold`/`adc_integral` valies shows no presence of `simides` on the same channel
- The noise tail shape matches the background shape for all varialbles
- This strongly suggests that the origin of the tail is a mistagging/detsim issue

## TP e- sample TP signal/noise distributions

In [ ]:
em_tpp = TrgPrimitivesPlotter(em_ws)

vars = {
    'adc_peak': {'bin_size': 10, 'xlim': (0, 400)},
    'samples_over_threshold': {'bin_size': 1, 'xlim': (0, 100)},
    'adc_integral': {'bin_size': 10, 'xlim': (0, 2000)},
}
for v, cfg in vars.items():
    h = em_tpp.make_var_hist(v, cfg['bin_size'])

    fig, axes = plt.subplots(1,3, figsize=(12,4))

    for rop_id in range(3):
        ax=axes[rop_id]

        hep.histplot(h[rop_id,sum,:], ax=ax, yerr=False, color='black', label='Total')
        hep.histplot(h[rop_id,1,:], ax=ax, yerr=False, histtype='fill', label='RadBkg', alpha=0.75)
        hep.histplot(h[rop_id,0,:], ax=ax, yerr=False, histtype='fill', label='Noise', alpha=0.75)
        ax.set_yscale('log')
        ax.set_xlim(*cfg['xlim'])
        ax.grid()
        ax.legend()
        ax.set_ylabel('Counts')
        
        ax.set_title(f"Readout plane {rop_id}")
    fig.suptitle(f"{v} distributions")

    fig.tight_layout()

### Observation

The noise TP distributions from the e- sample show similar features, although less prominent.
- the e- sample was produced in hydra mode, only the noise in TPCs where signal is present are simulated.


## Corrected purity tables

These are purity tables where the noise TP count is taken from the collection plane for all planes.

This is motivated by the expectation for the noise distribution to be the same across planes.
- To be confirmed

In [ ]:
ana.do_corrected_purity_tables()

### Observations

Correcting the purity with collection TP rates brings the numbers closer to expectations (is this actually correct?)

**Note**: At low thresholds (28-30), the correction introduces features in the background rates. Rates increase, before starting to decrease. This is due to normalisation issues.

At `adc_peak >= 38`, the fraction of noise TPs is 20-30%

# Conclusions

For MC production, `adc_peak >= 38` is a well motivated choice for all planes.
- Total rates in the 150 kHz ballpark
- Minimal "signal" losses compared to lower thresholds


# Test Code

In [ ]:
print(rad_tpp.make_generator_activity_table('adc_peak >= 44 & readout_plane_id == 2', norm='rate', geo_norm='crp'))
print(rad_tpp.make_generator_activity_table('adc_peak >= 44 & readout_plane_id == 2', norm='rate'))
# print(rad_tpp.make_generator_activity_table('adc_peak >= 44 & readout_plane_id ==2'))
# print(rad_tpp.make_generator_activity_table('adc_peak > 45 & readout_plane_id ==2 & samples_over_threshold > 8'))


In [ ]:
hep.histplot(rad_tpp.make_generator_counts_hist('adc_peak > 38')[:,2j])
plt.gca().set_yscale('log')

In [ ]:
fig = rad_tpp.plot_var_by_generator(query='adc_peak > 45', rop=2, var_spec={'name':'samples_over_threshold', 'bin_size':1, 'label': 'Samples Over Threshold'})
fig = rad_tpp.plot_var_by_generator(query='adc_peak > 45', rop=2, var_spec={'name':'adc_peak', 'bin_size':10, 'label': 'ADC Peak'})
fig = rad_tpp.plot_var_by_generator(query='adc_peak > 45 & samples_over_threshold > 9', rop=2, var_spec={'name':'adc_peak', 'bin_size':10, 'label': 'ADC Peak'})

# Don't Run Me

In [ ]:
print(tpp.make_generator_activity_table( "adc_peak >45 & readout_plane_id==0"))
print(tpp.make_generator_activity_table( "adc_peak >45 & readout_plane_id==1"))
print(tpp.make_generator_activity_table( "adc_peak >45 & readout_plane_id==2"))

In [ ]:
coll_tps = ws.tps.query('readout_view == 2 & sample_start >100 & sample_start < 8100 & bt_is_signal == 1 ')
coll_tps.extra_info.update({'readout_window': 8000})


In [ ]:
tps_by_gen = sorted([(n,df) for n,df in coll_tps.groupby('bt_generator_name')], reverse=True, key=lambda x: len(x[1]))


In [ ]:
counts = coll_tps.bt_generator_name.value_counts()
ax = counts.plot.bar()
ax.set_ylabel('counts')
ax.set_yscale('log')